In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 500)

In [ ]:
columnas_interes_alimentacion =  ["created_datetime", "last_modified_datetime", 
                     "product_name", "quantity", "brands", 
                     "categories_tags", "categories_en", "categories",
                     "labels", "labels_tags", "labels_en", "stores", "countries_en",
                     "ingredients_text", "ingredients_tags", "ingredients_analysis_tags",
                     "nutriscore_score", "nutriscore_grade", "nova_group", 
                     "food_groups", "food_groups_tags", "food_groups_en",
                     "environmental_score_score", "environmental_score_grade", "allergens_en",
                     "additives_tags", "additives_en", "packaging", "packaging_tags", "brand_owner",
                     "manufacturing_places_tags", "origins_en", "packaging_en"]

len(columnas_interes_alimentacion)

In [ ]:
# 1. Definimos la ruta correcta (con 'r' para raw string)
ruta = "./data/en.openfoodfacts.org.products.csv.gz"

lista_chunks_espana = []

print("🚀 Iniciando la lectura por bloques con columnas seleccionadas... Esto optimizará la memoria.")

# 3. Leemos el archivo en bloques, usando 'usecols' para ignorar el resto de columnas
bloques = pd.read_csv(
    ruta, 
    sep='\t', 
    chunksize=100000,        # Procesamos de 100k en 100k filas
    low_memory=False,
    usecols=columnas_interes_alimentacion  # <--- CLAVE: Solo carga estas columnas
)

for i, chunk in enumerate(bloques):
    # Filtramos el bloque actual: buscamos filas donde 'countries_en' contenga 'Spain'
    chunk_filtrado = chunk[chunk['countries_en'].str.contains("Spain", case=False, na=False)]
    
    # Si el bloque contiene productos de España, lo añadimos a la lista
    if not chunk_filtrado.empty:
        lista_chunks_espana.append(chunk_filtrado)
    
    # Imprime un contador cada 20 bloques para saber que el código sigue vivo
    if i % 20 == 0:
        filas_acumuladas = sum(len(df) for df in lista_chunks_espana)
        print(f"⏳ Bloques procesados: {i} | Filas de España acumuladas: {filas_acumuladas}")

# 4. Concatenamos todos los trozos filtrados en un único DataFrame final
df_alimentacion = pd.concat(lista_chunks_espana, ignore_index=True)

# 5. Limpieza básica opcional (recomendada tras concatenar)
# Eliminamos filas donde el nombre del producto esté vacío (suelen ser errores de escaneo)
df_alimentacion = df_alimentacion.dropna(subset=['product_name'])

print(f"\n✅ ¡Listo! Se ha generado 'df_alimentacion' con {df_alimentacion.shape[0]:,} filas y {df_alimentacion.shape[1]} columnas.")

In [ ]:
df_alimentacion.to_csv("./dataframes/alimentacion_df_inicial.csv", sep=',', index=False, encoding='utf-8')